# LeetCode 470 — Implement Rand10() Using Rand7() bằng Python

Link đề: https://leetcode.com/problems/implement-rand10-using-rand7/ · Độ khó: **Medium** · Chủ đề: Math, Rejection Sampling (lấy mẫu có loại bỏ), Probability and Statistics, Randomized

Skeleton LeetCode:

```python
# The rand7() API is already defined for you.
# def rand7():
# @return a random integer in the range 1 to 7

class Solution:
    def rand10(self):
        """
        :rtype: int
        """
        ...
```

Ghi chú: đề được viết lại bằng tiếng Việt theo đề gốc. Vì đây là bài **ngẫu nhiên**, notebook tự dựng `rand7()` có `seed`, kiểm tra bằng **thống kê chi-bình-phương** và bằng **đếm chính xác mọi khả năng** (phân số), không so với một đáp án cứng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có sẵn hàm `rand7()` trả về số nguyên **đồng đều** trong `1..7`; hãy viết `rand10()` trả về số **đồng đều** trong `1..10` (mỗi giá trị xác suất đúng `1/10`), **chỉ** được gọi `rand7()`.
2. Nắm khái niệm **phân phối đồng đều**, **kỳ vọng số lần gọi**, **lấy mẫu có loại bỏ (rejection sampling)** và vì sao `(rand7() + rand7()) % 10 + 1` **không** đều.
3. Dựng một **bộ chạy kiểm thử** cho bài ngẫu nhiên: `rand7` có `seed`, thống kê **chi-bình-phương** (cả giá trị đơn lẻ lẫn các cặp liên tiếp), và **chứng minh xác suất bằng đếm chính xác** (liệt kê mọi dãy kết quả của `rand7`, dùng `Fraction`).
4. Cài đặt 4 cách khác nhau về ý tưởng: **ghép 4 bit công bằng → hai lần gọi tạo `1..49` rồi loại `41..49` → tái sử dụng phần dư bị loại → phân rã `10 = 2 × 5`**, và so sánh số lần gọi `rand7` kỳ vọng.
5. Tránh các lỗi kinh điển: cộng hai `rand7`, quên `-1` khi ghép `(a-1)*7 + b`, loại sai ngưỡng (`< 40`), quên `+1` (ra `0..9`), và nhầm "kết quả trông có vẻ ngẫu nhiên" với "đều".

## 2. Đề bài & ví dụ

Cho API `rand7()` sinh một số nguyên **ngẫu nhiên đồng đều** trong khoảng `[1, 7]`. Hãy viết hàm `rand10()` sinh số nguyên ngẫu nhiên **đồng đều** trong khoảng `[1, 10]`.

- Chỉ được gọi API `rand7()`, **không** được gọi API ngẫu nhiên nào khác; **không** dùng hàm ngẫu nhiên có sẵn của ngôn ngữ.
- Mỗi test có một tham số **nội bộ** `n`: số lần `rand10()` được gọi khi chấm (không truyền vào hàm).

**Ví dụ 1:** `n = 1` → một kết quả như `[2]`.

**Ví dụ 2:** `n = 2` → ví dụ `[2, 8]`.

**Ví dụ 3:** `n = 3` → ví dụ `[3, 8, 10]`.

(Vì kết quả là ngẫu nhiên nên đầu ra trong đề chỉ là **minh họa**: mọi dãy với giá trị trong `1..10`, phân phối đồng đều, đều đúng.)

**Ràng buộc:** `1 <= n <= 10^5`.

**Follow up:**

- Kỳ vọng số lần gọi `rand7()` của bạn là bao nhiêu?
- Có thể tối thiểu hóa số lần gọi `rand7()` không?

In [1]:
import itertools
import math
import random
import time
from collections import Counter, defaultdict
from fractions import Fraction

# ---- Bộ dựng "rand7" của chúng ta (mô phỏng API của LeetCode) ----
_nguon = {"f": None}        # hàm sinh số 1..7 hiện tại
_dem = {"n": 0}             # đếm số lần rand7() bị gọi

def rand7():
    _dem["n"] += 1
    return _nguon["f"]()

def dung_rand7(seed):
    """Dựng nguồn rand7 đồng đều trong 1..7 với seed cố định (kết quả lặp lại được)."""
    rng = random.Random(seed)
    _nguon["f"] = lambda: rng.randint(1, 7)
    _dem["n"] = 0

dung_rand7(2024)
print([rand7() for _ in range(15)])            # lặp lại y hệt mỗi lần chạy vì có seed
dung_rand7(2024)
print([rand7() for _ in range(15)], "<- cùng seed, cùng dãy")

[4, 2, 6, 5, 3, 2, 6, 4, 7, 6, 7, 3, 5, 2, 6]
[4, 2, 6, 5, 3, 2, 6, 4, 7, 6, 7, 3, 5, 2, 6] <- cùng seed, cùng dãy


## 3. Kiến thức nền

### 3.1. Phân phối đồng đều và "đếm đủ"

"Đồng đều trên `1..7`" nghĩa là mỗi giá trị có xác suất `1/7`. Nếu chọn ngẫu nhiên đồng đều trong `N` khả năng thì mỗi khả năng xác suất `1/N`. Ta sẽ **đếm các khả năng** thay vì tính xác suất bằng công thức phức tạp.

In [2]:
for gia_tri in range(1, 8):
    print(gia_tri, Fraction(1, 7), end=" | ")
print()
print(sum(Fraction(1, 7) for _ in range(7)))       # 1: tổng xác suất phải bằng 1

1 1/7 | 2 1/7 | 3 1/7 | 4 1/7 | 5 1/7 | 6 1/7 | 7 1/7 | 
1


### 3.2. Hai lần gọi → một số trong `1..49` (hệ cơ số 7)

Gọi hai lần `a = rand7()`, `b = rand7()`: có `7 × 7 = 49` cặp `(a, b)`, **mỗi cặp xác suất `1/49`** (hai lần gọi độc lập). Biến cặp thành số: `x = (a - 1) * 7 + b`. Đây là **cách viết số hai chữ số trong hệ cơ số 7** (a-1 là chữ số hàng "bảy", b-1 là hàng đơn vị, cộng thêm 1). Vì vậy `x` chạy **đúng một lần** qua mỗi giá trị `1..49` (một-một), nên `x` **đồng đều** trong `1..49`.

In [3]:
cac_x = [(a - 1) * 7 + b for a in range(1, 8) for b in range(1, 8)]
print(cac_x[:10], "...", cac_x[-3:])
print(sorted(cac_x) == list(range(1, 50)))                 # True: mỗi giá trị 1..49 đúng một lần
print(Counter(cac_x).most_common(1))                       # [(1, 1)]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] ... [47, 48, 49]
True
[(1, 1)]


### 3.3. Phép `%` chia số thành các nhóm — và nhóm không đều

`x % 10` cho phần dư khi chia 10, nhận 10 giá trị `0..9`. Từ `1..49`, các số chia hết nhóm theo dư: dư 0 có `10, 20, 30, 40` (4 số), dư `1..9` có 5 số (ví dụ dư 1: `1, 11, 21, 31, 41`). **Không đều** vì `49` không chia hết cho `10`.

In [4]:
nhom = Counter(x % 10 for x in range(1, 50))
print(sorted(nhom.items()))            # dư 0 có 4 phần tử, các dư còn lại có 5
nhom40 = Counter(x % 10 for x in range(1, 41))
print(sorted(nhom40.items()))          # 1..40: mọi dư đều đúng 4 phần tử -> đều

[(0, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (6, 5), (7, 5), (8, 5), (9, 5)]
[(0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 4), (8, 4), (9, 4)]


### 3.4. Lấy mẫu có loại bỏ (rejection sampling)

Muốn đều trên `10` giá trị nhưng chỉ có nguồn đều trên `49`: **chỉ giữ lại 40 giá trị đầu** (`1..40`, chia hết cho 10), nếu rơi vào `41..49` thì **bỏ và thử lại**. Vì phần giữ lại vẫn đồng đều (mỗi giá trị còn lại cùng xác suất `1/49`), sau khi chuẩn hóa mỗi giá trị trong `1..40` xác suất `1/40`.

Lặp lại tới khi thành công là một **vòng `while True`**. Số lần thử là biến ngẫu nhiên **hình học**: nếu mỗi lần thành công với xác suất `p` thì kỳ vọng số lần thử là `1/p`.

In [5]:
p = Fraction(40, 49)
print("kỳ vọng số lần thử:", 1 / p, "=", float(1 / p))      # 49/40 = 1.225
print("kỳ vọng số lần gọi rand7:", 2 / p, "=", float(2 / p))  # 2.45 (mỗi lần thử gọi 2 lần)

kỳ vọng số lần thử: 49/40 = 1.225
kỳ vọng số lần gọi rand7: 49/20 = 2.45


### 3.5. Kiểm định chi-bình-phương (chi-square) tối thiểu

Gieo `N` lần, mỗi giá trị lý thuyết xuất hiện `e = N/10` lần. Với số lần đếm được `o_i`, thống kê là `χ² = Σ (o_i - e)² / e`. Nếu phân phối thật sự đều thì `χ²` xấp xỉ phân phối chi-bình-phương với `10 - 1 = 9` bậc tự do, kỳ vọng `9`. **Ngưỡng bác bỏ**: `χ² > 27.88` có xác suất chỉ `0.1%` nếu đều (mức ý nghĩa `α = 0.001`); `χ² > 21.67` là `1%`; `> 16.92` là `5%`. `χ²` càng lớn thì càng "lệch".

Với seed cố định, kết quả mô phỏng **lặp lại được**: test không "lúc đúng lúc sai".

In [6]:
def chi2(dem, k=10):
    """Thống kê chi-bình-phương cho đếm dem (Counter) trên các giá trị 1..k, kỳ vọng đều."""
    N = sum(dem.values())
    e = N / k
    return sum((dem[v] - e) ** 2 / e for v in range(1, k + 1))

NGUONG_9 = 27.877      # df = 9, alpha = 0.001

dem_deu = Counter({v: 10000 for v in range(1, 11)})
dem_lech = Counter({1: 12000, 2: 8000, **{v: 10000 for v in range(3, 11)}})
print(chi2(dem_deu), chi2(dem_lech))       # 0.0  và  (2000²/10000)*2 = 800

0.0 800.0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Thử đầu tiên (ngây thơ):** `(rand7() + rand7()) % 10 + 1`. Trông hợp lý? Hãy **đếm đủ 49 cặp**. Tổng `a + b` từ `2` đến `14`, nhưng các tổng **không đều**: tổng `8` xuất hiện 7 lần, tổng `2` chỉ 1 lần. Sau `% 10 + 1`, các giá trị chắc chắn **lệch**.

**Thử thứ hai:** dùng `x = (a - 1) * 7 + b` đều trên `1..49`, rồi `x % 10 + 1` cho mọi `x`. Đã thấy ở 3.3: dư `0` chỉ có 4 số còn dư khác có 5 số, nên kết quả `1` (ứng dư 0) xác suất `4/49` còn các giá trị khác `5/49`. **Lệch**.

**Cách đúng:** chỉ dùng `x` trong `1..40` (40 chia hết cho 10): mỗi giá trị `1..10` có đúng 4 số `x` ánh xạ tới (ví dụ giá trị `3`: `x = 3, 13, 23, 33`), xác suất mỗi giá trị là `4/40 = 1/10` sau khi chuẩn hóa. Nếu `x` rơi vào `41..49` (xác suất `9/49`) thì làm lại.

In [7]:
# Đếm chính xác 49 cặp cho hai thử nghiệm ngây thơ
dem_tong = Counter((a + b) % 10 + 1 for a in range(1, 8) for b in range(1, 8))
print("thử 1 (a+b)%10+1:", sorted(dem_tong.items()), "-> số khả năng /49")
dem_49 = Counter((x % 10) + 1 for x in range(1, 50))
print("thử 2 x%10+1 trên 1..49:", sorted(dem_49.items()))
dem_40 = Counter(((x - 1) % 10) + 1 for x in range(1, 41))
print("cách đúng (chỉ 1..40):", sorted(dem_40.items()))
assert len(set(dem_tong.values())) > 1 and len(set(dem_49.values())) > 1 and set(dem_40.values()) == {4}

thử 1 (a+b)%10+1: [(1, 5), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 5), (8, 6), (9, 7), (10, 6)] -> số khả năng /49
thử 2 x%10+1 trên 1..49: [(1, 4), (2, 5), (3, 5), (4, 5), (5, 5), (6, 5), (7, 5), (8, 5), (9, 5), (10, 5)]
cách đúng (chỉ 1..40): [(1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 4), (8, 4), (9, 4), (10, 4)]


## 5. Cách 1 — Brute Force / baseline: ghép 4 "đồng xu công bằng"

**Ý tưởng:** bắt đầu từ thứ ta biết chắc là đều: **một đồng xu 0/1 công bằng**. Có thể dựng từ `rand7()` bằng loại bỏ: gọi `rand7()`; nếu `7` thì **bỏ làm lại**; còn `1..6` thì `1,2,3 → 0` và `4,5,6 → 1` (mỗi bit xác suất `1/2` chính xác). Ghép **4 bit** thành số `0..15`: `x = 8*b3 + 4*b2 + 2*b1 + b0`; đều trên 16 giá trị. Nếu `x < 10` trả `x + 1`, ngược lại **bỏ và làm lại**.

In [8]:
def bit_cong_bang():
    while True:
        r = rand7()
        if r != 7:                    # bỏ 7 để còn 6 giá trị chia đôi được
            return (r - 1) // 3       # 1,2,3 -> 0 ; 4,5,6 -> 1

def rand10_bit():
    while True:
        x = bit_cong_bang() * 8 + bit_cong_bang() * 4 + bit_cong_bang() * 2 + bit_cong_bang()   # đều trên 0..15
        if x < 10:
            return x + 1

dung_rand7(1)
mau = [rand10_bit() for _ in range(20)]
print(mau)
assert all(1 <= v <= 10 for v in mau)

[5, 8, 6, 1, 7, 3, 10, 2, 6, 8, 10, 8, 5, 7, 6, 6, 8, 3, 10, 2]


In [9]:
class SolutionBit:
    def rand10(self):
        while True:
            x = 0
            for _ in range(4):
                while True:
                    r = rand7()
                    if r != 7:
                        break
                x = x * 2 + (r - 1) // 3
            if x < 10:
                return x + 1

dung_rand7(1)
print([SolutionBit().rand10() for _ in range(10)])

[5, 8, 6, 1, 7, 3, 10, 2, 6, 8]


**Độ phức tạp:** mỗi bit tốn kỳ vọng `7/6` lần gọi `rand7` (xác suất chấp nhận `6/7`); một lượt 4 bit: `4 × 7/6 = 14/3`; chấp nhận với xác suất `10/16` nên kỳ vọng `14/3 ÷ (10/16) = 112/15 ≈ 7.47` lần gọi. Thời gian **kỳ vọng** `O(1)` mỗi lần gọi `rand10`, bộ nhớ `O(1)`. (Trường hợp xấu nhất **không có chặn trên** vì có thể bị loại mãi; xác suất đó tiến về 0.)

**Nút thắt:** đúng và dễ chứng minh nhưng **phí** — `7.47` lần gọi trong khi ta chỉ cần "vừa đủ" `log₇(10) ≈ 1.18`. Phần lớn "độ ngẫu nhiên" bị vứt đi (bỏ giá trị `7`, bỏ `x ≥ 10`).

## 6. Ý tưởng thuật toán / bước đột phá

1. **Tạo một nguồn đồng đều đủ lớn rồi cắt đúng bội của 10.** Hai lần gọi cho một nguồn đều trên `49 = 7²` khả năng. Phần bội của 10 lớn nhất `<= 49` là `40`.
2. **Loại bỏ phần dư (`41..49`)**: bảo toàn tính đồng đều trên phần được giữ lại.
3. **Chứng minh xác suất bằng đếm:** mỗi lượt thành công với xác suất `40/49`, và *khi* thành công, mỗi giá trị `1..10` xuất hiện với xác suất `4/49`. Tổng xác suất cuối của giá trị `k` là `Σ_{r≥0} (9/49)^r · 4/49 = (4/49)/(1 - 9/49) = 1/10` (`r` = số lượt thất bại trước đó).
4. **Phần bị loại không nhất thiết vứt đi:** `41..49` là **9 giá trị còn đều** — vẫn là nguồn ngẫu nhiên có thể tái sử dụng (Cách 3), nên giảm số lần gọi.
5. **Phân rã:** `10 = 2 × 5`, ta có thể tạo riêng `rand2` và `rand5` từ `rand7` rồi ghép (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Hai lần gọi → `1..49`, loại `41..49` (cách chuẩn)

**Giải thích từng đoạn:**

- `a = rand7()`, `b = rand7()` → `x = (a - 1) * 7 + b` đều trên `1..49`.
- Nếu `x <= 40`: trả `(x - 1) % 10 + 1` (đều trên `1..10`, mỗi giá trị đúng 4 `x`).
- Ngược lại: lặp lại (vòng `while True`).

In [10]:
def rand10_hai_lan():
    while True:
        a = rand7()
        b = rand7()
        x = (a - 1) * 7 + b            # đều trên 1..49
        if x <= 40:
            return (x - 1) % 10 + 1    # 4 giá trị x cho mỗi kết quả

dung_rand7(2)
mau = [rand10_hai_lan() for _ in range(20)]
print(mau, "| số lần gọi rand7:", _dem["n"])
assert all(1 <= v <= 10 for v in mau)

[1, 3, 7, 10, 9, 4, 1, 9, 9, 6, 5, 8, 8, 6, 2, 9, 8, 10, 9, 3] | số lần gọi rand7: 54


In [11]:
class SolutionTwoCalls:
    def rand10(self):
        while True:
            x = (rand7() - 1) * 7 + rand7()      # 1..49 đều
            if x <= 40:
                return (x - 1) % 10 + 1

dung_rand7(2)
print([SolutionTwoCalls().rand10() for _ in range(10)])

[1, 3, 7, 10, 9, 4, 1, 9, 9, 6]


**Độ phức tạp (Cách 2):** mỗi lượt gọi `rand7` 2 lần, thành công với xác suất `40/49` ⇒ kỳ vọng `2 · 49/40 = 49/20 = 2.45` lần gọi `rand7`. Thời gian kỳ vọng `O(1)`, bộ nhớ `O(1)`.

### Cách 3 — Tái sử dụng phần dư bị loại

**Ý tưởng khác:** khi `x ∈ 41..49`, ta **biết** `x - 40` là một số **đều trên `1..9`** (9 giá trị ngang xác suất). Đó vẫn là "ngẫu nhiên có ích": gộp với **một lần gọi nữa** `b = rand7()` ta được `(a - 1) * 7 + b` với `a ∈ 1..9`, tức đều trên `1..63`. Giữ `1..60` (bội 10), nếu rơi vào `61..63` thì dư `3` giá trị đều (`1..3`); gộp thêm một lần gọi nữa được đều trên `1..21`, giữ `1..20`; nếu rơi vào `21` thì dư đúng 1 giá trị (không còn ngẫu nhiên) ⇒ làm lại từ đầu.

**Giải thích từng đoạn:** ba "tầng": tầng 1 dùng 2 lần gọi (đều `1..49`, giữ `1..40`); tầng 2 dùng phần dư `a = x - 40` (`1..9`) và 1 lần gọi (đều `1..63`, giữ `1..60`); tầng 3 dùng `a = x - 60` (`1..3`) và 1 lần gọi (đều `1..21`, giữ `1..20`).

In [12]:
def rand10_tai_su_dung():
    while True:
        a = rand7()
        b = rand7()
        x = (a - 1) * 7 + b            # 1..49
        if x <= 40:
            return (x - 1) % 10 + 1
        a = x - 40                      # 1..9, vẫn đều
        b = rand7()
        x = (a - 1) * 7 + b            # 1..63
        if x <= 60:
            return (x - 1) % 10 + 1
        a = x - 60                      # 1..3, vẫn đều
        b = rand7()
        x = (a - 1) * 7 + b            # 1..21
        if x <= 20:
            return (x - 1) % 10 + 1
        # còn x = 21: không còn ngẫu nhiên hữu ích -> làm lại từ đầu

dung_rand7(3)
mau = [rand10_tai_su_dung() for _ in range(20)]
print(mau, "| số lần gọi rand7:", _dem["n"])
assert all(1 <= v <= 10 for v in mau)

[2, 10, 9, 7, 9, 9, 8, 10, 3, 6, 6, 7, 3, 7, 6, 6, 8, 4, 9, 1] | số lần gọi rand7: 45


In [13]:
class SolutionReuse:
    def rand10(self):
        while True:
            a, b = rand7(), rand7()
            x = (a - 1) * 7 + b
            if x <= 40:
                return (x - 1) % 10 + 1
            a = x - 40
            x = (a - 1) * 7 + rand7()
            if x <= 60:
                return (x - 1) % 10 + 1
            a = x - 60
            x = (a - 1) * 7 + rand7()
            if x <= 20:
                return (x - 1) % 10 + 1

dung_rand7(3)
print([SolutionReuse().rand10() for _ in range(10)])

[2, 10, 9, 7, 9, 9, 8, 10, 3, 6]


**Độ phức tạp (Cách 3):** kỳ vọng gọi `rand7` thỏa `E = 2 + (9/49)(1 + (3/63)(1 + (1/21) E))` (cộng 2 lần đầu; với xác suất `9/49` thêm 1 lần, rồi với xác suất `3/63` thêm 1 lần nữa, rồi với xác suất `1/21` quay lại từ đầu). Giải ra `E ≈ 2.1933`, thấp hơn `2.45` của Cách 2. Thời gian kỳ vọng `O(1)`, bộ nhớ `O(1)`.

### Cách 4 — Phân rã `10 = 2 × 5`

**Ý tưởng khác:** thay vì nghĩ về `49`, hãy dựng hai nguồn nhỏ độc lập:

- `rand5()`: gọi `rand7()`, nếu `r <= 5` trả `r`, ngược lại làm lại (kỳ vọng `7/5` lần gọi).
- `rand2()`: gọi `rand7()`, nếu `r <= 6` trả `(r - 1) % 2 + 1` (lẻ → 1, chẵn → 2), ngược lại làm lại (kỳ vọng `7/6` lần gọi).

Ghép: `rand10 = (rand2() - 1) * 5 + rand5()`. Hai nguồn độc lập và đều nên cặp `(u, v)` đều trên `2 × 5 = 10` khả năng, và `(u - 1) * 5 + v` là một-một lên `1..10`.

In [14]:
def rand5():
    while True:
        r = rand7()
        if r <= 5:
            return r

def rand2():
    while True:
        r = rand7()
        if r <= 6:
            return (r - 1) % 2 + 1

def rand10_phan_ra():
    return (rand2() - 1) * 5 + rand5()

dung_rand7(4)
mau = [rand10_phan_ra() for _ in range(20)]
print(mau, "| số lần gọi rand7:", _dem["n"])
assert all(1 <= v <= 10 for v in mau)

[8, 4, 7, 1, 4, 3, 2, 5, 3, 6, 2, 3, 2, 8, 3, 5, 4, 2, 7, 8] | số lần gọi rand7: 56


In [15]:
class SolutionDecompose:
    def rand10(self):
        while True:
            r = rand7()
            if r <= 6:
                u = (r - 1) % 2          # 0 hoặc 1, đều
                break
        while True:
            r = rand7()
            if r <= 5:
                return u * 5 + r         # 1..10 đều

dung_rand7(4)
print([SolutionDecompose().rand10() for _ in range(10)])

[8, 4, 7, 1, 4, 3, 2, 5, 3, 6]


**Độ phức tạp (Cách 4):** kỳ vọng `7/6 + 7/5 = 77/30 ≈ 2.567` lần gọi `rand7`. Thời gian kỳ vọng `O(1)`, bộ nhớ `O(1)`. Ưu điểm: **rất dễ chứng minh** (hai nguồn độc lập, đều). Nhược điểm: tốn hơn Cách 2 và 3.

> Tóm tắt kỳ vọng số lần gọi: Cách 1 `≈ 7.47`; Cách 4 `≈ 2.567`; Cách 2 `= 2.45`; Cách 3 `≈ 2.193`. (Mục 11 tính chính xác bằng phân số; mục 12 kiểm tra bằng mô phỏng.)

## 8. TRACE từng bước

Để trace một thuật toán ngẫu nhiên, ta **thay `rand7` bằng một dãy kịch bản** (tape) có sẵn rồi xem thuật toán phản ứng thế nào.

**Cách 2 với tape `[7, 7, 2, 4]`:**

| lượt | `a` | `b` | `x = (a-1)*7 + b` | `x <= 40`? | kết quả |
|---|---|---|---|---|---|
| 1 | 7 | 7 | `6*7 + 7 = 49` | không | bỏ, làm lại |
| 2 | 2 | 4 | `1*7 + 4 = 11` | có | `(11 - 1) % 10 + 1 = 1` |

**Cách 3 với tape `[7, 6, 5]`:** lượt 1: `a=7, b=6` → `x = 48 > 40` ⇒ phần dư `a = 48 - 40 = 8` (đều trên `1..9`); gọi tiếp `b = 5` → `x = 7*7 + 5 = 54 <= 60` ⇒ trả `(54 - 1) % 10 + 1 = 4`. Chỉ tốn 3 lần gọi thay vì phải bỏ 2 lần đã gọi.

In [16]:
def chay_voi_tape(ham, tape):
    """Chạy ham() với rand7 lấy lần lượt từ tape, in từng lần gọi."""
    it = iter(tape)
    stt = [0]
    def nguon():
        stt[0] += 1
        v = next(it)
        print(f"  rand7() lần {stt[0]} -> {v}")
        return v
    _nguon["f"] = nguon
    _dem["n"] = 0
    try:
        kq = ham()
    except StopIteration:
        kq = None
        print("  (hết tape)")
    print(f"  => kết quả {kq}, đã gọi rand7 {_dem['n']} lần")
    return kq

print("Cách 2, tape [7, 7, 2, 4]:")
assert chay_voi_tape(SolutionTwoCalls().rand10, [7, 7, 2, 4]) == 1
print("Cách 3, tape [7, 6, 5]:")
assert chay_voi_tape(SolutionReuse().rand10, [7, 6, 5]) == 4

Cách 2, tape [7, 7, 2, 4]:
  rand7() lần 1 -> 7
  rand7() lần 2 -> 7
  rand7() lần 3 -> 2
  rand7() lần 4 -> 4
  => kết quả 1, đã gọi rand7 4 lần
Cách 3, tape [7, 6, 5]:
  rand7() lần 1 -> 7
  rand7() lần 2 -> 6
  rand7() lần 3 -> 5
  => kết quả 4, đã gọi rand7 3 lần


In [17]:
# Bảng ánh xạ của Cách 2: x trong 1..40 -> kết quả (mỗi kết quả đúng 4 giá trị x)
for k in range(1, 11):
    print(f"kết quả {k:>2}: x = {[x for x in range(1, 41) if (x - 1) % 10 + 1 == k]}")
print("Cách 4, tape [3, 4]  (rand2: r=3 -> u=0; rand5: r=4 <= 5 nên nhận -> 0*5 + 4):")
assert chay_voi_tape(SolutionDecompose().rand10, [3, 4]) == 4
print("Cách 4, tape [7, 2, 6, 3]  (7 bị loại ở rand2; r=2 -> u=1; 6 bị loại ở rand5; r=3 nhận):")
assert chay_voi_tape(SolutionDecompose().rand10, [7, 2, 6, 3]) == 8

kết quả  1: x = [1, 11, 21, 31]
kết quả  2: x = [2, 12, 22, 32]
kết quả  3: x = [3, 13, 23, 33]
kết quả  4: x = [4, 14, 24, 34]
kết quả  5: x = [5, 15, 25, 35]
kết quả  6: x = [6, 16, 26, 36]
kết quả  7: x = [7, 17, 27, 37]
kết quả  8: x = [8, 18, 28, 38]
kết quả  9: x = [9, 19, 29, 39]
kết quả 10: x = [10, 20, 30, 40]
Cách 4, tape [3, 4]  (rand2: r=3 -> u=0; rand5: r=4 <= 5 nên nhận -> 0*5 + 4):
  rand7() lần 1 -> 3
  rand7() lần 2 -> 4
  => kết quả 4, đã gọi rand7 2 lần
Cách 4, tape [7, 2, 6, 3]  (7 bị loại ở rand2; r=2 -> u=1; 6 bị loại ở rand5; r=3 nhận):
  rand7() lần 1 -> 7
  rand7() lần 2 -> 2
  rand7() lần 3 -> 6
  rand7() lần 4 -> 3
  => kết quả 8, đã gọi rand7 4 lần


## 9. Code nộp LeetCode

Dùng Cách 2 (hai lần gọi, loại `41..49`): ngắn nhất, dễ chứng minh, kỳ vọng `2.45` lần gọi. (Nếu muốn tối ưu số lần gọi theo follow-up, dùng Cách 3.)

Lưu ý: trên LeetCode `rand7` đã được định nghĩa sẵn ở phạm vi toàn cục; trong notebook ta đã dựng `rand7` giả lập ở trên.

In [18]:
class Solution:
    def rand10(self):
        """
        :rtype: int
        """
        while True:
            i = rand7() - 1            # 0..6
            j = rand7()                # 1..7
            x = i * 7 + j              # đều trên 1..49 (cơ số 7)
            if x <= 40:                # chỉ giữ bội của 10 để các kết quả đều nhau
                return (x - 1) % 10 + 1


def chay_ban_test(n, seed, cls=None):
    """Mô phỏng cách LeetCode chấm: gọi rand10() n lần với rand7 có seed, trả về danh sách."""
    dung_rand7(seed)
    sol = (cls or Solution)()
    return [sol.rand10() for _ in range(n)]

for n in (1, 2, 3):
    kq = chay_ban_test(n, seed=n)
    print(f"n = {n} -> {kq}")
    assert len(kq) == n and all(1 <= v <= 10 for v in kq)

n = 1 -> [2]
n = 2 -> [1, 3]
n = 3 -> [2, 10, 9]


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đếm).** Hai lần gọi độc lập cho 49 cặp `(a, b)` đồng khả năng. Ánh xạ `(a, b) → x = (a-1)*7 + b` là một-một lên `{1, ..., 49}`, nên `x` đều trên `1..49`.

**Bổ đề 2 (điều kiện thành công).** Gọi `A` là biến cố "`x <= 40`" (xác suất `40/49`). Với mỗi `k ∈ 1..10`, đúng 4 giá trị `x ∈ 1..40` cho kết quả `k` (`x ≡ k (mod 10)`), nên `P(kết quả = k và thành công ở lượt này) = 4/49`.

**Định lý.** Xác suất `rand10() = k` là `1/10` cho mọi `k`.

*Chứng minh.* Kết quả là `k` ở lượt thứ `r + 1` khi `r` lượt đầu đều thất bại (mỗi lượt xác suất `9/49`, các lượt độc lập) rồi lượt này cho `k`:

`P(k) = Σ_{r=0}^{∞} (9/49)^r · (4/49) = (4/49) / (1 - 9/49) = (4/49)/(40/49) = 1/10`. ∎

**Thuật toán dừng với xác suất 1:** xác suất còn chạy sau `r` lượt là `(9/49)^r → 0`. Mục 12 **tính chính xác bằng phân số**: liệt kê mọi dãy kết quả của `rand7` tới độ sâu `D`, khối lượng xác suất của từng giá trị bằng nhau **chính xác** (không phải xấp xỉ), phần "chưa kết thúc" tiến về 0.

**Vì sao `(a + b) % 10 + 1` sai:** các tổng không đồng khả năng (tổng 8 có 7 cặp, tổng 2 chỉ 1 cặp); ánh xạ không "chia đều" 49 khả năng.

**Vì sao `x % 10 + 1` trên toàn `1..49` sai:** `49 = 4*10 + 9` nên 9 giá trị đầu của dãy dư được 5 khả năng, giá trị còn lại 4 khả năng.

## 11. Phân tích Time / Space Complexity

**Thời gian kỳ vọng mỗi lần gọi `rand10()`**, tính theo số lần gọi `rand7`:

| Cách | Kỳ vọng số lần gọi `rand7` | Thời gian | Space |
|---|---|---|---|
| 1. Ghép 4 bit | `112/15 ≈ 7.47` | `O(1)` kỳ vọng | `O(1)` |
| 2. Hai lần, loại `41..49` | `49/20 = 2.45` | `O(1)` kỳ vọng | `O(1)` |
| 3. Tái sử dụng phần dư | `≈ 2.1933` | `O(1)` kỳ vọng | `O(1)` |
| 4. Phân rã `2 × 5` | `77/30 ≈ 2.567` | `O(1)` kỳ vọng | `O(1)` |

**Trường hợp xấu nhất không có chặn trên** (có thể bị loại mãi): xác suất cần ít nhất `r` lượt là `(9/49)^(r-1)` ở Cách 2, giảm rất nhanh. Với `n` lần gọi, tổng thời gian kỳ vọng `O(n)`.

**Chặn dưới lý thuyết:** mỗi lần gọi cho `log₂ 7 ≈ 2.807` bit ngẫu nhiên; cần `log₂ 10 ≈ 3.322` bit, nên tối thiểu `≈ 1.183` lần gọi; các thuật toán không-nhớ (bỏ phần dư) không đạt được mức đó, nhưng có thể tiến sát bằng cách **giữ lại phần dư giữa các lần gọi `rand10`** (ngoài phạm vi LeetCode).

In [19]:
# Tính chính xác kỳ vọng số lần gọi bằng phân số
E1 = Fraction(4 * 7, 6) / Fraction(10, 16)
E2 = Fraction(2) / Fraction(40, 49)
# E = c + k*E  =>  E = c / (1 - k)
c = 2 + Fraction(9, 49) * (1 + Fraction(3, 63))
k = Fraction(9, 49) * Fraction(3, 63) * Fraction(1, 21)
E3 = c / (1 - k)
E4 = Fraction(7, 6) + Fraction(7, 5)
for ten, e in [("Cách 1", E1), ("Cách 2", E2), ("Cách 3", E3), ("Cách 4", E4)]:
    print(f"{ten}: E = {e} = {float(e):.4f}")
assert E1 == Fraction(112, 15) and E2 == Fraction(49, 20) and E4 == Fraction(77, 30)
assert E3 < E2 < E4 < E1

Cách 1: E = 112/15 = 7.4667
Cách 2: E = 49/20 = 2.4500
Cách 3: E = 329/150 = 2.1933
Cách 4: E = 77/30 = 2.5667


## 12. Bộ test

Với bài ngẫu nhiên ta không so với một đáp án cứng. Ta kiểm tra **ba loại tính chất**:

1. **Miền giá trị:** mọi kết quả thuộc `1..10`, đúng `n` kết quả.
2. **Thống kê (mô phỏng có seed):** chi-bình-phương cho các giá trị đơn lẻ, cho các **cặp liên tiếp** (độc lập), và **kỳ vọng số lần gọi** đo được so với lý thuyết.
3. **Đếm chính xác:** liệt kê mọi dãy kết quả có thể của `rand7` rồi cộng xác suất bằng `Fraction` (không có sai số), để thấy mỗi giá trị `1..10` có xác suất **bằng nhau tuyệt đối**.

### 12.1. Miền giá trị, kích thước, tái lập theo seed

In [20]:
tat_ca = [("Cách 1", SolutionBit), ("Cách 2 (nộp)", Solution), ("Cách 3", SolutionReuse), ("Cách 4", SolutionDecompose)]

for ten, cls in tat_ca:
    for n in (1, 2, 3, 1000):
        kq = chay_ban_test(n, seed=n + 7, cls=cls)
        assert len(kq) == n and all(isinstance(v, int) and 1 <= v <= 10 for v in kq), (ten, n)
    # cùng seed => cùng kết quả (tái lập)
    assert chay_ban_test(500, seed=99, cls=cls) == chay_ban_test(500, seed=99, cls=cls)
    # khác seed => (gần như chắc chắn) khác kết quả
    assert chay_ban_test(500, seed=99, cls=cls) != chay_ban_test(500, seed=100, cls=cls)
print("Miền 1..10, độ dài n, tái lập theo seed: OK cho cả 4 cách")

kq = chay_ban_test(100_000, seed=1)                  # n tối đa của đề
assert len(kq) == 100_000 and set(kq) == set(range(1, 11))
print("n = 100000: đủ cả 10 giá trị")

Miền 1..10, độ dài n, tái lập theo seed: OK cho cả 4 cách


n = 100000: đủ cả 10 giá trị


### 12.2. Thống kê chi-bình-phương cho từng giá trị

`N = 100 000` lần gọi với mỗi cách (seed cố định). Mỗi giá trị kỳ vọng `10 000` lần; `χ²` phải **dưới ngưỡng 27.877** (`α = 0.001`).

In [21]:
N = 100_000
for ten, cls in tat_ca:
    dem = Counter(chay_ban_test(N, seed=12345, cls=cls))
    x2 = chi2(dem)
    print(f"{ten:<14} chi2 = {x2:6.2f}  (ngưỡng {NGUONG_9})  đếm = {[dem[v] for v in range(1, 11)]}")
    assert x2 < NGUONG_9, (ten, x2)

Cách 1         chi2 =   7.16  (ngưỡng 27.877)  đếm = [10015, 9889, 10197, 9981, 9965, 10007, 9946, 9936, 9963, 10101]
Cách 2 (nộp)   chi2 =   8.40  (ngưỡng 27.877)  đếm = [10093, 9968, 10058, 10022, 10059, 9967, 9837, 10031, 10120, 9845]
Cách 3         chi2 =  12.44  (ngưỡng 27.877)  đếm = [10166, 10027, 10065, 9970, 10195, 9894, 10009, 9867, 9955, 9852]


Cách 4         chi2 =   8.81  (ngưỡng 27.877)  đếm = [10094, 9843, 9993, 9848, 10100, 10071, 9948, 10114, 10012, 9977]


**Kiểm tra độ nhạy của bài test:** một lời giải **sai** phải bị bắt. Với `(rand7() + rand7()) % 10 + 1` hoặc `x % 10 + 1` trên toàn `1..49`, `χ²` phải **vượt xa** ngưỡng.

In [22]:
class SaiTong:
    def rand10(self):
        return (rand7() + rand7()) % 10 + 1

class SaiMod49:
    def rand10(self):
        return ((rand7() - 1) * 7 + rand7()) % 10 + 1

for ten, cls in [("(a+b)%10+1", SaiTong), ("x%10+1 trên 1..49", SaiMod49)]:
    dem = Counter(chay_ban_test(N, seed=12345, cls=cls))
    x2 = chi2(dem)
    print(f"{ten:<20} chi2 = {x2:9.2f}  đếm = {[dem[v] for v in range(1, 11)]}")
    assert x2 > 10 * NGUONG_9      # lệch rất xa so với ngưỡng

(a+b)%10+1           chi2 =   4485.95  đếm = [10327, 8188, 8095, 8193, 8185, 8156, 10242, 12135, 14306, 12173]


x%10+1 trên 1..49    chi2 =    430.71  đếm = [8046, 10289, 10171, 10237, 10268, 10283, 10093, 10054, 10267, 10292]


### 12.3. Chi-bình-phương trung bình qua nhiều seed (kiểm tra không "may mắn")

Nếu phân phối đều thì `χ²` trung bình qua nhiều lần chạy độc lập phải xấp xỉ **9** (kỳ vọng của phân phối chi-bình-phương 9 bậc tự do). Chạy 200 seed, mỗi seed 5000 lần, và đếm số lần vượt các ngưỡng.

In [23]:
for ten, cls in tat_ca:
    cac_x2 = [chi2(Counter(chay_ban_test(5000, seed=s, cls=cls))) for s in range(200)]
    tb = sum(cac_x2) / len(cac_x2)
    vuot_5 = sum(v > 16.919 for v in cac_x2)
    print(f"{ten:<14} chi2 trung bình = {tb:5.2f} (lý thuyết 9);  số seed vượt ngưỡng 5%: {vuot_5}/200 (lý thuyết ~10)")
    assert 7.5 < tb < 10.5
    assert vuot_5 < 30

Cách 1         chi2 trung bình =  8.59 (lý thuyết 9);  số seed vượt ngưỡng 5%: 4/200 (lý thuyết ~10)


Cách 2 (nộp)   chi2 trung bình =  9.02 (lý thuyết 9);  số seed vượt ngưỡng 5%: 12/200 (lý thuyết ~10)


Cách 3         chi2 trung bình =  9.01 (lý thuyết 9);  số seed vượt ngưỡng 5%: 10/200 (lý thuyết ~10)


Cách 4         chi2 trung bình =  9.26 (lý thuyết 9);  số seed vượt ngưỡng 5%: 14/200 (lý thuyết ~10)


### 12.4. Tính độc lập của các lần gọi liên tiếp

"Đều" từng lần chưa đủ: các lần gọi liên tiếp phải **độc lập**. Ta xếp kết quả thành các cặp không chồng lấn `(r1, r2), (r3, r4), ...` và kiểm tra `100` ô `(r1, r2)` đều nhau: `χ²` với `99` bậc tự do, ngưỡng `α = 0.001` là khoảng `148.2`.

In [24]:
NGUONG_99 = 148.23
for ten, cls in tat_ca:
    day = chay_ban_test(200_000, seed=777, cls=cls)
    cap = Counter(zip(day[0::2], day[1::2]))
    e = len(day) // 2 / 100
    x2 = sum((cap[(a, b)] - e) ** 2 / e for a in range(1, 11) for b in range(1, 11))
    print(f"{ten:<14} chi2(cặp) = {x2:6.1f}  (ngưỡng {NGUONG_99})")
    assert x2 < NGUONG_99

Cách 1         chi2(cặp) =  102.1  (ngưỡng 148.23)
Cách 2 (nộp)   chi2(cặp) =   92.4  (ngưỡng 148.23)


Cách 3         chi2(cặp) =  104.6  (ngưỡng 148.23)


Cách 4         chi2(cặp) =   90.1  (ngưỡng 148.23)


### 12.5. Kỳ vọng số lần gọi `rand7`: mô phỏng so với lý thuyết

In [25]:
ly_thuyet = {"Cách 1": E1, "Cách 2 (nộp)": E2, "Cách 3": E3, "Cách 4": E4}
for ten, cls in tat_ca:
    dung_rand7(31415)
    sol = cls()
    for _ in range(200_000):
        sol.rand10()
    tb = _dem["n"] / 200_000
    print(f"{ten:<14} đo được {tb:.4f}  lý thuyết {float(ly_thuyet[ten]):.4f}")
    assert abs(tb - float(ly_thuyet[ten])) < 0.05

Cách 1         đo được 7.4632  lý thuyết 7.4667


Cách 2 (nộp)   đo được 2.4539  lý thuyết 2.4500
Cách 3         đo được 2.1952  lý thuyết 2.1933


Cách 4         đo được 2.5685  lý thuyết 2.5667


### 12.6. Chứng minh bằng đếm chính xác (liệt kê mọi dãy kết quả của `rand7`)

Mô phỏng chỉ cho xác suất xấp xỉ. Để **chứng minh bằng đếm**, ta chạy thuật toán với một **tape** (dãy giá trị `rand7` cho trước). Nếu thuật toán cần nhiều giá trị hơn tape thì ta **rẽ nhánh**: thử cả 7 giá trị tiếp theo. Mỗi nhánh kết thúc sau `L` lần gọi có xác suất đúng `7^(-L)`. Ta cộng **số nguyên** (đơn vị `7^(-D)`) rồi so sánh; phần chưa kết thúc sau độ sâu `D` được ghi riêng.

In [26]:
class HetBang(Exception):
    pass

def liet_ke(thuat_toan, D):
    """Trả (tong, chua): tong[v] = trọng số (đơn vị 7^-D) của các nhánh kết thúc <= D lần gọi cho kết quả v;
    chua = trọng số các nhánh chưa kết thúc sau D lần gọi."""
    tong = defaultdict(int)
    chua = 0
    ngan_xep = [()]
    while ngan_xep:
        tien_to = ngan_xep.pop()
        vt = [0]
        def nguon():
            if vt[0] >= len(tien_to):
                raise HetBang
            v = tien_to[vt[0]]
            vt[0] += 1
            return v
        _nguon["f"] = nguon
        try:
            kq = thuat_toan()
        except HetBang:
            if len(tien_to) == D:
                chua += 1
            else:
                ngan_xep.extend(tien_to + (v,) for v in range(1, 8))
            continue
        tong[kq] += 7 ** (D - len(tien_to))
    return dict(tong), chua

# Thử với thuật toán ngây thơ: đếm chính xác 49 khả năng
tong, chua = liet_ke(lambda: (rand7() + rand7()) % 10 + 1, 2)
print("ngây thơ (a+b)%10+1, trên 49:", sorted(tong.items()), "chưa xong:", chua)
assert sum(tong.values()) + chua == 49 and len(set(tong.values())) > 1

ngây thơ (a+b)%10+1, trên 49: [(1, 5), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4), (7, 5), (8, 6), (9, 7), (10, 6)] chưa xong: 0


In [27]:
cac_thuat_toan = {
    "Cách 1 (4 bit)": (lambda: SolutionBit().rand10(), 6),
    "Cách 2 (hai lần)": (lambda: Solution().rand10(), 6),
    "Cách 3 (tái sử dụng)": (lambda: SolutionReuse().rand10(), 6),
    "Cách 4 (phân rã)": (lambda: SolutionDecompose().rand10(), 6),
}
for ten, (ham, D) in cac_thuat_toan.items():
    tong, chua = liet_ke(ham, D)
    tong_so = 7 ** D
    assert sum(tong.values()) + chua == tong_so                 # không mất nhánh nào
    assert set(tong) == set(range(1, 11))                       # kết quả luôn thuộc 1..10
    gia_tri = set(tong.values())
    xs = {v: Fraction(m, tong_so) for v, m in tong.items()}
    print(f"{ten:<22} D={D}: mỗi giá trị khối lượng {sorted(gia_tri)}/{tong_so} ; chưa xong = {Fraction(chua, tong_so)} ≈ {chua / tong_so:.5f}")
    assert len(gia_tri) == 1, "các giá trị phải CÓ KHỐI LƯỢNG BẰNG NHAU TUYỆT ĐỐI"

Cách 1 (4 bit)         D=6: mỗi giá trị khối lượng [7047]/117649 ; chưa xong = 47179/117649 ≈ 0.40101
Cách 2 (hai lần)       D=6: mỗi giá trị khối lượng [11692]/117649 ; chưa xong = 729/117649 ≈ 0.00620
Cách 3 (tái sử dụng)   D=6: mỗi giá trị khối lượng [11764]/117649 ; chưa xong = 9/117649 ≈ 0.00008
Cách 4 (phân rã)       D=6: mỗi giá trị khối lượng [11727]/117649 ; chưa xong = 379/117649 ≈ 0.00322


Mỗi giá trị `1..10` có **cùng khối lượng chính xác** ở mọi độ sâu ta liệt kê, trong khi phần chưa kết thúc giảm theo cấp số nhân khi `D` tăng (rất nhỏ với Cách 2, 3, 4; còn lớn ở Cách 1 vì mỗi lượt đã cần tối thiểu 4 lần gọi). Vì mỗi giá trị luôn nhận đúng cùng khối lượng `m` và tổng bằng `1 - chưa`, ta có `P(k) = (1 - chưa)/10 → 1/10` khi `D → ∞`: **khớp công thức đóng** ở mục 10. Thử tính giá trị giới hạn bằng phân số cho Cách 2:

In [28]:
# P(k) = (4/49) * sum_{r>=0} (9/49)^r  : tổng cấp số nhân = 1/(1 - 9/49)
p_k = Fraction(4, 49) / (1 - Fraction(9, 49))
print("P(k) =", p_k)
assert p_k == Fraction(1, 10)

# Kiểm tra: tổng riêng đến r = 30 tiến sát 1/10 từ dưới lên
tong_rieng = sum(Fraction(4, 49) * Fraction(9, 49) ** r for r in range(31))
print(float(tong_rieng), "< 0.1 :", tong_rieng < p_k, "; sai số", float(p_k - tong_rieng))

# Với từng độ sâu, khối lượng của một giá trị trong Cách 2 đúng bằng (4/49) * sum_{r<D/2} (9/49)^r
for D in (2, 4, 6):
    tong, chua = liet_ke(lambda: Solution().rand10(), D)
    mong_doi = Fraction(4, 49) * sum(Fraction(9, 49) ** r for r in range(D // 2))
    assert {Fraction(m, 7 ** D) for m in tong.values()} == {mong_doi}
print("Khối lượng mỗi giá trị khớp công thức hình học đúng tại D = 2, 4, 6")

P(k) = 1/10
0.1 < 0.1 : True ; sai số 1.5326370208633666e-24
Khối lượng mỗi giá trị khớp công thức hình học đúng tại D = 2, 4, 6


### 12.7. Edge case

`n = 1` (một lần gọi), seed khác nhau cho các giá trị khác nhau, và bắt trường hợp "luôn bị loại" bằng tape: ta kiểm tra thuật toán **không trả lời sớm sai** khi rơi vào vùng bị loại.

In [29]:
# n = 1 trên nhiều seed: luôn trong 1..10 và xuất hiện đủ 10 giá trị
cac_gia_tri = {chay_ban_test(1, seed=s)[0] for s in range(300)}
print(sorted(cac_gia_tri))
assert cac_gia_tri == set(range(1, 11))

# Vùng bị loại: (7,7) -> x = 49 > 40: phải gọi tiếp chứ không được trả lời
def thu_tape(cls, tape):
    it = iter(tape)
    _nguon["f"] = lambda: next(it)
    _dem["n"] = 0
    return cls().rand10(), _dem["n"]

assert thu_tape(Solution, [7, 7, 1, 1]) == (1, 4)          # x = 49 bị loại, rồi x = 1 -> 1
assert thu_tape(Solution, [6, 6, 1, 1]) == (1, 4)          # x = 5*7 + 6 = 41 > 40 bị loại
assert thu_tape(Solution, [6, 5, 1, 1]) == (10, 2)         # x = 40 -> (40 - 1) % 10 + 1 = 10 : biên được giữ
assert thu_tape(Solution, [1, 1]) == (1, 2)                # x = 1

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


Cách 3 xử lý vùng bị loại bằng cách **giữ phần dư**; ta chạy với tape để xem từng bước:

In [30]:
print("Cách 3 với tape [7, 7, 1] :")
kq = chay_voi_tape(SolutionReuse().rand10, [7, 7, 1])
# x = 6*7 + 7 = 49 > 40 -> a = 9 ; b = 1 -> x = 8*7 + 1 = 57 <= 60 -> (57 - 1) % 10 + 1 = 7
assert kq == 7

Cách 3 với tape [7, 7, 1] :
  rand7() lần 1 -> 7
  rand7() lần 2 -> 7
  rand7() lần 3 -> 1
  => kết quả 7, đã gọi rand7 3 lần


## 13. Các lỗi người mới hay gặp

Mỗi lỗi ở đây ta **chứng minh bằng đếm chính xác** (mục 12.6): kết quả "lệch" không chỉ là cảm giác.

### Lỗi 1 — Cộng hai lần gọi: `(rand7() + rand7()) % 10 + 1`

Tổng của hai biến đều **không** đều (phân phối hình tam giác).

In [31]:
tong, chua = liet_ke(lambda: (rand7() + rand7()) % 10 + 1, 2)
print({v: Fraction(m, 49) for v, m in sorted(tong.items())})
print("lớn nhất / nhỏ nhất:", max(tong.values()), "/", min(tong.values()), " (đáng lẽ bằng nhau)")
assert max(tong.values()) != min(tong.values())

{1: Fraction(5, 49), 2: Fraction(4, 49), 3: Fraction(4, 49), 4: Fraction(4, 49), 5: Fraction(4, 49), 6: Fraction(4, 49), 7: Fraction(5, 49), 8: Fraction(6, 49), 9: Fraction(1, 7), 10: Fraction(6, 49)}
lớn nhất / nhỏ nhất: 7 / 4  (đáng lẽ bằng nhau)


**Vì sao sai:** tổng `a + b` có 7 cách cho giá trị `8` nhưng chỉ 1 cách cho giá trị `2`. **Cách đúng:** ghép `(a - 1) * 7 + b` (một-một lên `1..49`) rồi **loại bỏ** để còn bội của 10.

### Lỗi 2 — Quên `- 1` khi ghép: `x = rand7() * 7 + rand7()`

Khi đó `x` chạy từ `8` đến `56`, không phải `1..49`: ngưỡng `x <= 40` giữ `33` giá trị, **không chia hết cho 10**.

In [32]:
class Sai_QuenTru1:
    def rand10(self):
        while True:
            x = rand7() * 7 + rand7()          # SAI: 8..56
            if x <= 40:
                return (x - 1) % 10 + 1

def phan_phoi_chinh_xac(cls, D=4):
    tong, chua = liet_ke(lambda: cls().rand10(), D)
    return tong, chua

tong, chua = phan_phoi_chinh_xac(Sai_QuenTru1)
print("khối lượng các giá trị (đơn vị 7^-4):", sorted(tong.items()))
assert len(set(tong.values())) > 1
cac_x = [a * 7 + b for a in range(1, 8) for b in range(1, 8)]
print("x chạy từ", min(cac_x), "đến", max(cac_x), "số giá trị x <= 40:", sum(v <= 40 for v in cac_x))

khối lượng các giá trị (đơn vị 7^-4): [(1, 195), (2, 195), (3, 195), (4, 195), (5, 195), (6, 195), (7, 195), (8, 260), (9, 260), (10, 260)]
x chạy từ 8 đến 56 số giá trị x <= 40: 33


**Vì sao sai:** có `33` giá trị `x ≤ 40` (từ `8`), nên các kết quả nhận `3` hoặc `4` giá trị `x` — không đều. **Cách đúng:** `(rand7() - 1) * 7 + rand7()`.

### Lỗi 3 — Ngưỡng sai: `x < 40` (loại thừa một giá trị)

Chỉ giữ `39` giá trị `1..39` ⇒ kết quả `10` (ứng `x = 40`) chỉ có `3` giá trị `x`, còn lại có `4`.

In [33]:
class Sai_Nguong:
    def rand10(self):
        while True:
            x = (rand7() - 1) * 7 + rand7()
            if x < 40:                          # SAI: phải là <= 40
                return (x - 1) % 10 + 1

tong, chua = phan_phoi_chinh_xac(Sai_Nguong)
print(sorted(tong.items()))
assert tong[10] < tong[1]
dem = Counter(chay_ban_test(N, seed=5, cls=Sai_Nguong))
print("chi2 =", round(chi2(dem), 1), "vượt ngưỡng:", chi2(dem) > NGUONG_9)
assert chi2(dem) > NGUONG_9

[(1, 236), (2, 236), (3, 236), (4, 236), (5, 236), (6, 236), (7, 236), (8, 236), (9, 236), (10, 177)]


chi2 = 628.6 vượt ngưỡng: True


**Vì sao sai:** ngưỡng phải là **bội của 10** đúng bằng `40` (`<= 40`), thiếu hay thừa 1 đều lệch. **Cách đúng:** `x <= 40`.

### Lỗi 4 — Quên `+ 1`: kết quả `0..9`

`x % 10` cho `0..9`, giá trị `10` không bao giờ xuất hiện và có thêm giá trị `0` không hợp lệ.

In [34]:
class Sai_QuenCong1:
    def rand10(self):
        while True:
            x = (rand7() - 1) * 7 + rand7()
            if x <= 40:
                return x % 10                   # SAI: 0..9

kq = chay_ban_test(1000, seed=1, cls=Sai_QuenCong1)
print("min/max:", min(kq), max(kq), "-> có 0:", 0 in kq, "; thiếu 10:", 10 not in kq)
assert min(kq) == 0 and 10 not in kq

min/max: 0 9 -> có 0: True ; thiếu 10: True


**Vì sao sai:** đề yêu cầu `1..10`. **Cách đúng:** `x % 10 + 1` (khi `x` chạy `0..39`) hoặc `(x - 1) % 10 + 1` (khi `x` chạy `1..40`).

### Lỗi 5 (khái niệm) — "Trông ngẫu nhiên" không có nghĩa là "đều"

In 20 giá trị ra nhìn chung đều "lộn xộn" ở cả cách đúng lẫn cách sai. Chỉ **đếm** (nhiều lần), hoặc **liệt kê chính xác**, mới phát hiện lệch.

In [35]:
print("đúng :", chay_ban_test(20, seed=8, cls=Solution))
print("sai  :", chay_ban_test(20, seed=8, cls=SaiMod49), "<- nhìn không ra lệch, cần thống kê")

đúng : [10, 3, 3, 1, 9, 1, 6, 5, 5, 6, 4, 2, 3, 8, 2, 5, 6, 8, 7, 4]
sai  : [1, 4, 4, 2, 10, 8, 2, 7, 6, 6, 7, 5, 3, 4, 4, 9, 3, 6, 7, 9] <- nhìn không ra lệch, cần thống kê


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "dựng nguồn ngẫu nhiên đều với kích thước `m` từ nguồn đều `k`" → **rejection sampling**: tạo nguồn đủ lớn rồi loại bỏ phần dư.
- **Công thức ghép chữ số:** `x = (a - 1) * k + b` biến hai biến đều trên `1..k` thành một biến đều trên `1..k²`. Ghép nhiều lần cho `k³`, `k⁴`...
- **Chọn ngưỡng:** bội lớn nhất của `m` không vượt quá `k^t`; xác suất thành công là `bội / k^t`.
- **Đừng cộng hoặc nhân** các biến đều rồi lấy dư: phân phối không đều.
- **Tối ưu số lần gọi** bằng cách **giữ lại phần dư còn đều** (Cách 3).
- **Cách kiểm thử:** (1) kiểm tra miền, (2) chi-bình-phương với seed, (3) nếu có thể, **liệt kê chính xác** để chứng minh.
- **Khi phân rã được** (như `10 = 2 × 5`) thì chứng minh gọn nhất, vì dùng nguồn độc lập.

## 15. Biến thể

**Biến thể A — `rand7()` từ `rand10()`:** ghép hai lần `rand10` thành `1..100`, giữ `1..98` (bội của 7), trả `(x - 1) % 7 + 1`; xác suất thành công `98/100`.

**Biến thể B — tổng quát `randN()` từ `randM()`:** vòng lặp "ghép chữ số, nhân khoảng, loại phần dư, **giữ phần dư**" cho bất kỳ `M, N`.

**Biến thể C — số lần gọi tối thiểu:** giữ phần dư **qua các lần gọi** `rand10()` (trạng thái toàn cục) để tiệm cận `log_M N` lần gọi mỗi lần.

In [36]:
def rand7_tu_rand10(rand10_fn):
    while True:
        x = (rand10_fn() - 1) * 10 + rand10_fn()      # đều trên 1..100
        if x <= 98:
            return (x - 1) % 7 + 1

def rand_n_tu_rand_m(n, m, rand_m_fn):
    """Biến thể B: randN từ randM, giữ phần dư (thuật toán "mang theo khoảng")."""
    gia_tri, khoang = 0, 1                     # gia_tri đều trên [0, khoang)
    while True:
        while khoang < n:
            gia_tri = gia_tri * m + (rand_m_fn() - 1)
            khoang *= m
        bien = khoang - khoang % n              # bội lớn nhất của n không vượt quá khoang
        if gia_tri < bien:
            return gia_tri % n + 1
        gia_tri -= bien                         # phần dư còn đều trên [0, khoang - bien)
        khoang -= bien

# Kiểm tra A: chạy rand7_tu_rand10 trên nguồn rand10 xây từ Solution
dung_rand7(55)
sol = Solution()
dem7 = Counter(rand7_tu_rand10(sol.rand10) for _ in range(70_000))
x2 = sum((dem7[v] - 10_000) ** 2 / 10_000 for v in range(1, 8))
print("rand7 từ rand10: chi2 =", round(x2, 2), " (df=6, ngưỡng 0.001 = 22.46)", sorted(dem7.items()))
assert x2 < 22.46

# Kiểm tra B: rand10 từ rand7 tổng quát
dung_rand7(56)
dem10 = Counter(rand_n_tu_rand_m(10, 7, rand7) for _ in range(100_000))
print("randN từ randM (10 từ 7): chi2 =", round(chi2(dem10), 2), "| số lần gọi trung bình:", _dem["n"] / 100_000)
assert chi2(dem10) < NGUONG_9
assert set(dem10) == set(range(1, 11))

rand7 từ rand10: chi2 = 2.09  (df=6, ngưỡng 0.001 = 22.46) [(1, 10062), (2, 10003), (3, 9981), (4, 10047), (5, 9881), (6, 10013), (7, 10013)]
randN từ randM (10 từ 7): chi2 = 9.71 | số lần gọi trung bình: 2.19525


## 16. So sánh các cách làm

| Cách | Ý tưởng | Kỳ vọng gọi `rand7` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép 4 bit | Đồng xu công bằng từ `rand7`, 4 bit → `0..15`, loại `>= 10` | `≈ 7.47` | `O(1)` | Dễ chứng minh nhưng phí; để hiểu |
| 2. Hai lần → `1..49`, loại `41..49` | Ghép cơ số 7, giữ bội của 10 | `2.45` | `O(1)` | **Cách nộp bài**: ngắn, đúng, chuẩn |
| 3. Tái sử dụng phần dư | Giữ phần dư còn đều để ghép tiếp | `≈ 2.193` | `O(1)` | Follow-up: giảm số lần gọi |
| 4. Phân rã `2 × 5` | `rand2` và `rand5` độc lập rồi ghép | `≈ 2.567` | `O(1)` | Khi muốn chứng minh cực gọn |

In [37]:
for ten, cls in tat_ca:
    dung_rand7(2468)
    t0 = time.perf_counter()
    sol = cls()
    for _ in range(100_000):
        sol.rand10()
    t = time.perf_counter() - t0
    print(f"{ten:<14} {1000 * t:7.1f} ms cho 100000 lần; gọi rand7 trung bình {_dem['n'] / 100_000:.3f} lần/lần")

Cách 1           475.5 ms cho 100000 lần; gọi rand7 trung bình 7.471 lần/lần
Cách 2 (nộp)     106.6 ms cho 100000 lần; gọi rand7 trung bình 2.452 lần/lần
Cách 3            94.0 ms cho 100000 lần; gọi rand7 trung bình 2.194 lần/lần


Cách 4           175.2 ms cho 100000 lần; gọi rand7 trung bình 2.567 lần/lần


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `x = (a-1)*7 + b` với `a = 3, b = 5`; (b) với `a = 7, b = 7` thì `Solution().rand10()` làm gì ở lượt đó; (c) `(40 - 1) % 10 + 1`; (d) số giá trị `x` trong `1..40` cho kết quả `7`; (e) xác suất `x <= 40`.

**Bài 2 (tính tay).** Với thuật toán của Cách 2 và tape `[2, 3, 6, 6, 4, 5]`, viết ra từng lượt `(a, b, x)` và kết quả đầu tiên được trả về, cùng số lần gọi `rand7`.

**Bài 3 (điền TODO).** Hoàn thành `rand10_todo()` theo Cách 2.

In [38]:
def rand10_todo():
    # TODO 1: while True:
    # TODO 2: a = rand7(); b = rand7(); x = (a - 1) * 7 + b
    # TODO 3: nếu x <= 40: return (x - 1) % 10 + 1
    return None

# KHÔNG gọi rand10_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `rand10_phan_ra_todo()` theo Cách 4 (`rand2` rồi `rand5`) **mà không nhìn lại** mục 7. Kiểm tra bằng `chi2` với seed cố định.

**Bài 5 (biến thể / mini challenge).** Viết `rand7_tu_rand10_ans()` (Biến thể A: từ `rand10` sang `rand7`; coi `Solution().rand10` là nguồn) rồi tính **chính xác** kỳ vọng số lần gọi `rand10` của nó. Gợi ý: mỗi lượt gọi 2 lần, thành công với xác suất `98/100`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `2*7 + 5 = 19`; (b) `x = 49 > 40`, bỏ và gọi tiếp; (c) `10`; (d) `4` giá trị: `7, 17, 27, 37`; (e) `40/49`. Bài 3: dùng đúng 3 dòng gợi ý.
- **Mức 2 (vừa):** Bài 2: lượt 1 `(2, 3)` → `x = 10 <= 40` ⇒ trả `(10 - 1) % 10 + 1 = 10` ngay, chỉ tốn 2 lần gọi. Bài 4: `rand2` bỏ `7`; `rand5` bỏ `6, 7`.
- **Mức 3 (sâu):** Bài 5: kỳ vọng `2 / (98/100) = 100/49`. Để kiểm tra, dùng `chi2` với `k = 7` bậc tự do 6 (ngưỡng `22.46` cho `α = 0.001`).

## 19. Đáp án tham khảo cho bài tập

In [39]:
# Bài 1
assert (3 - 1) * 7 + 5 == 19
assert (40 - 1) % 10 + 1 == 10
assert len([x for x in range(1, 41) if (x - 1) % 10 + 1 == 7]) == 4
print("(e) xác suất x <= 40 =", Fraction(40, 49))
print("(b): với (7, 7) -> x = 49 bị loại, gọi tiếp ->", thu_tape(Solution, [7, 7, 1, 1]))
print("Bài 1 OK")

# Bài 2
kq, so_goi = thu_tape(Solution, [2, 3, 6, 6, 4, 5])
print("Bài 2: kết quả", kq, "sau", so_goi, "lần gọi")
assert (kq, so_goi) == (10, 2)

# Bài 3
def rand10_todo_ans():
    while True:
        a = rand7()
        b = rand7()
        x = (a - 1) * 7 + b
        if x <= 40:
            return (x - 1) % 10 + 1

dung_rand7(10)
dem = Counter(rand10_todo_ans() for _ in range(100_000))
assert chi2(dem) < NGUONG_9 and set(dem) == set(range(1, 11))
tong, chua = liet_ke(rand10_todo_ans, 4)
assert len(set(tong.values())) == 1
print("Bài 3 OK, chi2 =", round(chi2(dem), 2))

# Bài 4
def rand10_phan_ra_todo():
    while True:
        r = rand7()
        if r <= 6:
            u = (r - 1) % 2
            break
    while True:
        r = rand7()
        if r <= 5:
            return u * 5 + r

dung_rand7(11)
dem = Counter(rand10_phan_ra_todo() for _ in range(100_000))
assert chi2(dem) < NGUONG_9 and set(dem) == set(range(1, 11))
tong, chua = liet_ke(rand10_phan_ra_todo, 5)
assert len(set(tong.values())) == 1
print("Bài 4 OK, chi2 =", round(chi2(dem), 2))

# Bài 5
def rand7_tu_rand10_ans(rand10_fn):
    while True:
        x = (rand10_fn() - 1) * 10 + rand10_fn()
        if x <= 98:
            return (x - 1) % 7 + 1

dung_rand7(12)
sol = Solution()
dem7 = Counter(rand7_tu_rand10_ans(sol.rand10) for _ in range(70_000))
x2 = sum((dem7[v] - 10_000) ** 2 / 10_000 for v in range(1, 8))
assert x2 < 22.46 and set(dem7) == set(range(1, 8))
E5 = Fraction(2) / Fraction(98, 100)
print("Bài 5: kỳ vọng số lần gọi rand10 =", E5, "=", float(E5), "; chi2 =", round(x2, 2))
assert E5 == Fraction(100, 49)
# đếm chính xác trên nguồn rand10 giả định đều: 100 cặp, 98 được giữ, mỗi kết quả 14 cặp
assert Counter((x - 1) % 7 + 1 for x in range(1, 99)) == Counter({v: 14 for v in range(1, 8)})
print("Bài 5 OK")

(e) xác suất x <= 40 = 40/49
(b): với (7, 7) -> x = 49 bị loại, gọi tiếp -> (1, 4)
Bài 1 OK
Bài 2: kết quả 10 sau 2 lần gọi
Bài 3 OK, chi2 = 3.26


Bài 4 OK, chi2 = 4.12
Bài 5: kỳ vọng số lần gọi rand10 = 100/49 = 2.0408163265306123 ; chi2 = 3.4
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (rejection sampling: đều trên `m` từ nguồn đều trên `k`):**

```python
while True:
    i = rand7() - 1          # 0..6
    j = rand7()              # 1..7
    x = i * 7 + j            # đều trên 1..49
    if x <= 40:              # 40 = bội lớn nhất của 10 không vượt quá 49
        return (x - 1) % 10 + 1
```

**Checklist:**

- [ ] Ghép hai lần gọi bằng `(a - 1) * 7 + b` (không cộng, không nhân rồi lấy dư).
- [ ] Ngưỡng là **bội của 10** đúng bằng `40` và dùng `x <= 40` (không `< 40`).
- [ ] Khi bị loại thì **lặp lại** (không trả lời tạm), nên có `while True`.
- [ ] Kết quả `(x - 1) % 10 + 1` nằm trong `1..10` (có `+ 1`).
- [ ] Chỉ gọi `rand7()`, không dùng module `random` trong lời giải nộp.
- [ ] Biết kỳ vọng số lần gọi của cách mình chọn (`2.45` cho Cách 2) và cách giảm nó (tái sử dụng phần dư).
- [ ] Kiểm thử bằng chi-bình-phương với seed (hoặc đếm chính xác), không chỉ in vài giá trị.
- [ ] Đúng signature `class Solution` / `rand10(self)` (không có tham số truyền vào).